# OFFBALL 04 — Detección de anomalías económicas
**Proyecto académico de IA y Big Data · 25/11/2026**

## Objetivo
Identificar valoraciones inusualmente bajas respecto a la estimación de R05, con R02 como comprobación complementaria. El resultado es una lista de candidatos para revisión, no una valoración verdadera, un precio de fichaje ni una recomendación de compra.

Reutilizamos predicciones temporales exportadas del notebook 03. No ajustamos modelos sobre el test ni alteramos los modelos originales. R02 y R05 comparten variables y datos: su concordancia no es una validación independiente. Los nombres R02/R05 corresponden a las configuraciones sin probabilidades GMM; GMM permanece como módulo deportivo separado.

**Entrada:** `OFFBALL_Integracion_Final.zip` de la entrega 24/11, o el ZIP de entrega completa que lo contiene. En Colab, ejecuta las celdas y sube uno de esos ZIP cuando se solicite.

In [ ]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
import os, json, hashlib, importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
SALIDA = ROOT / 'OFFBALL_04_resultados'
SALIDA.mkdir(exist_ok=True)
R02 = 'R02_datos_GMM'
R05 = 'R05_datos_GMM_log'
RANGOS = ['Menos de 1M', 'Entre 1M y 5M', '5M o mas']
PERCENTIL = 0.95
PRESUPUESTO_EUR = 5_000_000  # valor publicado, no precio de traspaso
MAX_DIAS_STATS = 365
MAX_DIAS_VALORACION = 180
MIN_CALIBRACION = 100

# Decisiones exploratorias, no umbrales validados como eficacia de scouting.
# Ajustar parámetros solo con un nuevo conjunto de validación para una evaluación futura.

## 1. Carga y comprobación de las predicciones
El CSV de validación contiene predicciones de folds temporales de 2021–2023, pero no identificadores individuales. El CSV de test sí incluye `player_id`, fecha de valoración y cierre de temporada. Se comprueban los rangos temporales, las claves y la positividad de los valores antes de calcular logaritmos.

In [ ]:
ruta_env = os.environ.get('OFFBALL_INPUT')
if ruta_env:
    entrada = Path(ruta_env)
else:
    archivos = sorted(ROOT.glob('OFFBALL_Integracion_Final*.zip')) or sorted(ROOT.glob('OFFBALL_Entrega_Completa*.zip'))
    if not archivos:
        from google.colab import files
        archivos = [Path(n) for n in files.upload() if n.endswith('.zip')]
    assert archivos, 'Sube el ZIP de integración o de entrega completa.'
    entrada = archivos[0]

with ZipFile(entrada) as exterior:
    if 'predicciones_test_completas.csv' in exterior.namelist():
        contenido = entrada.read_bytes()
    else:
        internos = [n for n in exterior.namelist() if Path(n).name == 'OFFBALL_Integracion_Final.zip']
        assert len(internos) == 1, 'No se encontró un ZIP de integración final único.'
        contenido = exterior.read(internos[0])
with ZipFile(BytesIO(contenido)) as z:
    validacion = pd.read_csv(z.open('predicciones_validacion.csv'))
    test = pd.read_csv(z.open('predicciones_test_completas.csv'), parse_dates=['valuation_date','fin_temporada'])
    config_origen = json.load(z.open('configuracion.json'))
sha256 = hashlib.sha256(contenido).hexdigest()

val = validacion.loc[validacion.modelo.eq(R05), ['anio','valor_real','valor_estimado']].copy()
test = test[['player_id','valuation_date','fin_temporada','market_value_in_eur',R02,R05]].copy()
assert len(val) and len(test)
assert not test.duplicated(['player_id','valuation_date']).any()
assert test[['player_id','valuation_date','fin_temporada']].notna().all().all()
for df, cols in [(val,['valor_real','valor_estimado']),(test,['market_value_in_eur',R02,R05])]:
    assert np.isfinite(df[cols]).all().all() and df[cols].gt(0).all().all()
assert val.anio.max() < test.valuation_date.dt.year.min()
assert test.fin_temporada.lt(test.valuation_date).all()
assert len(test) == config_origen['n_test']
print(f'Validación R05: {len(val):,} valoraciones ({val.anio.min()}–{val.anio.max()})')
print(f'Test: {len(test):,} valoraciones de {test.player_id.nunique():,} jugadores')
print('Periodo test:', test.valuation_date.min().date(), 'a', test.valuation_date.max().date())
print('SHA256 del ZIP de integración:', sha256)

## 2. Calibración histórica del residuo
Definimos el residuo logarítmico $r=\ln(\hat V/V)$, donde $V$ es la valoración publicada y $\hat V$ la estimación. Un residuo positivo indica una estimación superior a la publicada. La transformación logarítmica permite comparar errores relativos.

Se estima el percentil 95 de todos los residuos R05 de cada rango de valor observado en validación. La estratificación controla parcialmente la sobreestimación de jugadores baratos. No se interpreta el percentil como probabilidad de infravaloración. Las valoraciones repetidas pueden estar correlacionadas; la calibración es a nivel de valoración, no jugador. Tampoco es un intervalo de predicción ni garantiza una tasa de alertas del 5% en otro periodo.

In [ ]:
def rango_economico(serie):
    return pd.cut(serie, bins=[0,1e6,5e6,np.inf], labels=RANGOS, right=False)

def residuo_log(observado, estimado):
    return np.log(estimado / observado)

val['rango'] = rango_economico(val.valor_real)
val['residuo_log'] = residuo_log(val.valor_real, val.valor_estimado)
calibracion = val.groupby('rango', observed=True).residuo_log.agg(n='size', mediana='median')
assert len(calibracion) == len(RANGOS) and calibracion.n.ge(MIN_CALIBRACION).all()
calibracion['umbral_log'] = val.groupby('rango', observed=True).residuo_log.quantile(PERCENTIL)
calibracion['ratio_umbral'] = np.exp(calibracion.umbral_log)
display(calibracion.round(3))
# Diagnóstico de estabilidad anual, no se usa para ajustar el test.
estabilidad = val.groupby(['anio','rango'], observed=True).residuo_log.agg(n='size', p95=lambda s:s.quantile(PERCENTIL)).reset_index()
display(estabilidad.round(3))

## 3. Aplicación al test retrospectivo
Se mantienen fijos los umbrales de validación. Una alerta económica exige superar el umbral del rango y que R05 estime un valor mayor al observado. R02 se utiliza después como comprobación complementaria. La evaluación es retrospectiva: los modelos originales ya se compararon en este test, de modo que no equivale a una validación externa intacta.

In [ ]:
test['rango'] = rango_economico(test.market_value_in_eur)
test['residuo_log'] = residuo_log(test.market_value_in_eur,test[R05])
test['umbral_log'] = test.rango.map(calibracion.umbral_log).astype(float)
assert test.umbral_log.notna().all()
test['exceso_log'] = test.residuo_log - test.umbral_log
test['diferencia_eur'] = test[R05] - test.market_value_in_eur
test['diferencia_pct'] = 100 * test.diferencia_eur / test.market_value_in_eur
test['alerta_economica'] = test.residuo_log.gt(0) & test.exceso_log.gt(0)
test['acuerdo_r02'] = test[R02].gt(test.market_value_in_eur)
test['dias_desde_temporada'] = (test.valuation_date-test.fin_temporada).dt.days
fecha_corte = test.valuation_date.max()
test['dias_desde_valoracion'] = (fecha_corte-test.valuation_date).dt.days
resumen_rango = test.groupby('rango', observed=True).alerta_economica.agg(valoraciones='size',alertas='sum',tasa='mean')
resumen_anual = test.groupby(test.valuation_date.dt.year).alerta_economica.agg(valoraciones='size',alertas='sum',tasa='mean')
display(resumen_rango.round(4),resumen_anual.round(4))
print('Alertas históricas:', int(test.alerta_economica.sum()), 'de',len(test))

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
for i,rango in enumerate(RANGOS):
    x=val.loc[val.rango.eq(rango),'residuo_log']
    ax.hist(x,bins=45,density=True,histtype='step',linewidth=1.7,label=rango)
    ax.axvline(calibracion.loc[rango,'umbral_log'],ls='--',alpha=.65)
ax.set(xlabel='Residuo logarítmico',ylabel='Densidad',title='Calibración histórica R05 por rango económico')
ax.legend();fig.tight_layout();fig.savefig(SALIDA/'01_calibracion.png',dpi=140);plt.show()

## 4. Lista de candidatos por futbolista
Conservamos primero la última valoración disponible de cada jugador (evitamos elegir retrospectivamente su mejor anomalía). Luego filtramos por alerta, presupuesto publicado inferior a 5 M€, concordancia positiva de R02 y actualidad de la información: cierre de temporada a no más de 365 días de su valoración y valoración a no más de 180 días del último día presente en el test. Son reglas operativas propuestas, no estimaciones de eficacia.

Ordenamos por exceso sobre el umbral del propio rango. Esta puntuación no representa una probabilidad ni permite inferir el potencial deportivo. La fecha de corte es la del archivo, no el día actual; la lista no acredita disponibilidad actual. Los CSV no contienen nombres ni clubes, que deberán añadirse mediante un join por `player_id` con datos verificados.

In [ ]:
ultimos = test.sort_values(['valuation_date','player_id']).drop_duplicates('player_id',keep='last').copy()
alerta = ultimos.alerta_economica
presupuesto = alerta & ultimos.market_value_in_eur.lt(PRESUPUESTO_EUR)
acuerdo = presupuesto & ultimos.acuerdo_r02
vigencia = acuerdo & ultimos.dias_desde_temporada.le(MAX_DIAS_STATS) & ultimos.dias_desde_valoracion.le(MAX_DIAS_VALORACION)
candidatos = ultimos.loc[vigencia].sort_values(['exceso_log','diferencia_eur','player_id'],ascending=[False,False,True]).copy()
candidatos.insert(0,'ranking',np.arange(1,len(candidatos)+1))
embudo = pd.DataFrame({'etapa':['Última valoración por jugador','Alerta económica','Dentro del presupuesto','Acuerdo R02','Actualidad de los datos'], 'jugadores':[len(ultimos),int(alerta.sum()),int(presupuesto.sum()),int(acuerdo.sum()),int(vigencia.sum())]})
print('Fecha de corte histórica:', fecha_corte.date())
display(embudo)
cols=['ranking','player_id','valuation_date','market_value_in_eur',R05,R02,'diferencia_eur','diferencia_pct','exceso_log','dias_desde_temporada']
display(candidatos[cols].round(3))
assert candidatos.player_id.is_unique
assert candidatos.market_value_in_eur.lt(PRESUPUESTO_EUR).all()
assert candidatos.exceso_log.gt(0).all()

In [ ]:
fig, ax = plt.subplots(figsize=(9,4))
ax.barh(embudo.etapa[::-1],embudo.jugadores[::-1])
for i,n in enumerate(embudo.jugadores[::-1]): ax.text(n+max(embudo.jugadores)*.012,i,str(n),va='center')
ax.set(xlim=(0,max(embudo.jugadores)*1.15),xlabel='Jugadores únicos',title='Embudo de selección retrospectiva')
fig.tight_layout();fig.savefig(SALIDA/'02_embudo.png',dpi=140);plt.show()

## 5. Sensibilidad del umbral (diagnóstico)
Comparamos 90, 95 y 99 percentiles para observar cómo varía el volumen de alertas. No escogemos un umbral nuevo a partir del test. Para evaluar si se detectan oportunidades reales, necesitaríamos información posterior independiente (evolución de valor, traspasos, rendimiento y revisión humana), además de comparar con candidatos no señalados.

In [ ]:
sensibilidad = []
for q in [.90,.95,.99]:
    umbrales = val.groupby('rango',observed=True).residuo_log.quantile(q)
    u = ultimos.rango.map(umbrales).astype(float)
    senal = ultimos.residuo_log.gt(0) & ultimos.residuo_log.gt(u)
    filtro = senal & ultimos.acuerdo_r02 & ultimos.market_value_in_eur.lt(PRESUPUESTO_EUR) & ultimos.dias_desde_temporada.le(MAX_DIAS_STATS) & ultimos.dias_desde_valoracion.le(MAX_DIAS_VALORACION)
    sensibilidad.append({'percentil':q,'alertas_ultimos':int(senal.sum()),'candidatos_filtrados':int(filtro.sum())})
sensibilidad = pd.DataFrame(sensibilidad)
display(sensibilidad)

## 6. Exportación reproducible
Se exporta la calibración (reutilizable para futuras inferencias), predicciones con alertas, candidatos y diagnósticos. El JSON de candidatos contiene tipos compatibles con una futura importación en MongoDB, pero no realiza ninguna escritura. No requiere volver a entrenar Random Forest o GMM.

In [ ]:
parametros = {'modelo_principal':R05,'modelo_complementario':R02,'percentil':PERCENTIL,'presupuesto_eur_exclusivo':PRESUPUESTO_EUR,'max_dias_stats':MAX_DIAS_STATS,'max_dias_valoracion':MAX_DIAS_VALORACION,'min_calibracion_por_rango':MIN_CALIBRACION,'fecha_corte':str(fecha_corte.date()),'sha256_integracion':sha256,'evaluacion':'retrospectiva_exploratoria','umbrales_log':{str(k):float(v) for k,v in calibracion.umbral_log.items()}}
metricas = {'n_validacion':len(val),'n_test':len(test),'n_jugadores':len(ultimos),'n_alertas_test':int(test.alerta_economica.sum()),'tasa_alertas_test':float(test.alerta_economica.mean()),'n_alertas_ultimos':int(alerta.sum()),'n_candidatos':len(candidatos)}
for nombre, tabla in {'calibracion':calibracion.reset_index(),'estabilidad_calibracion':estabilidad,'resumen_rango':resumen_rango.reset_index(),'resumen_anual':resumen_anual.rename_axis('anio').reset_index(),'predicciones_test_anomalias':test,'candidatos':candidatos,'embudo':embudo,'sensibilidad':sensibilidad}.items():
    tabla.to_csv(SALIDA/f'{nombre}.csv',index=False)
candidatos.to_json(SALIDA/'candidatos.json',orient='records',date_format='iso',force_ascii=False)
(SALIDA/'configuracion_anomalias.json').write_text(json.dumps(parametros,indent=2,ensure_ascii=False),encoding='utf-8')
(SALIDA/'metricas_anomalias.json').write_text(json.dumps(metricas,indent=2),encoding='utf-8')
print(json.dumps(metricas,indent=2))

## 7. MLflow
Se registra un nuevo experimento para mantener la trazabilidad de esta etapa. No modifica la base SQLite original de integración. En Colab se instala MLflow si no está presente. En otros entornos, instalar con `pip install mlflow` y ejecutar la celda. No se registran precision/recall de infravaloración porque no hay etiquetas independientes.

In [ ]:
import subprocess, sys
if importlib.util.find_spec('mlflow') is None and 'google.colab' in sys.modules:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','mlflow'])
if importlib.util.find_spec('mlflow') is not None:
    import mlflow
    from mlflow.tracking import MlflowClient
    mlflow.set_tracking_uri(f"sqlite:///{(SALIDA/'anomalias_mlflow.db').resolve().as_posix()}")
    experiment_name = 'OFFBALL_04_Anomalias_Economicas'
    client = MlflowClient()
    experimento = client.get_experiment_by_name(experiment_name)
    exp_id = experimento.experiment_id if experimento else client.create_experiment(experiment_name,artifact_location=(SALIDA/'artefactos_mlflow').resolve().as_uri())
    mlflow.set_experiment(experiment_id=exp_id)
    with mlflow.start_run(run_name='R05_P95_R02_confirmacion') as run:
        mlflow.log_params({k:v for k,v in parametros.items() if k != 'umbrales_log'})
        mlflow.log_params({f'umbral_{i}':v for i,v in enumerate(parametros['umbrales_log'].values())})
        mlflow.log_metrics(metricas)
        for p in SALIDA.iterdir():
            if p.is_file() and p.suffix in {'.csv','.json','.png'}:
                mlflow.log_artifact(str(p))
        print('MLflow run:',run.info.run_id)
else:
    print('MLflow no instalado: registro pendiente. Los CSV, JSON y métricas sí se han generado.')

## Conclusión y siguientes pasos
La señal mide una discrepancia respecto al comportamiento histórico del modelo, no una anomalía económica confirmada. La muestra solo cubre la cohorte de atacantes del notebook 03, con valoraciones y estadísticas históricas. No extrapolar a porteros, defensas o a todos los jugadores sin validación. La antigüedad de la fuente y la aproximación al cierre de temporada se heredan de la etapa anterior. El valor publicado puede omitir lesiones, contratos, ligas y circunstancias de mercado.

Siguientes pasos: enriquecer candidatos con identidad y perfil GMM deportivo mediante claves/fechas compatibles, revisar los casos manualmente, validar el criterio en un periodo posterior no utilizado para seleccionar modelos y conectar los resultados a MongoDB y FastAPI. No se declara que ninguna de estas tareas esté completada.